# Readability


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
import language
from analysis import (MACRO, ORDER, benjamini_hochberg, bootstrap_paired,
                      permutation_paired, publish, pvalue, write_captions)
from language import FLOOR, NEEDS_LENGTH, target_grade
from settings import RESULTS_DIR, measure_column

pd.set_option('display.width', 220, 'display.max_columns', 40)

# Model Names
NAME = dict(zip(['gpt-5.6-luna', 'claude-haiku-4-5-20251001',
                 'gemini-3.5-flash-lite', 'deepseek-v4-flash',
                 'mistral-small-2603', 'gemma4:31b-cloud'], ORDER))

FORMULAS = [measure_column(name) for name in NEEDS_LENGTH]
PRIMARY = ['fkgl', 'aae']
SECONDARY = ['fre', 'mean_aoa', 'response_length']
STATED_MINOR = [7, 9, 11, 13, 15, 17]
STATED_ADULT = [18, 21]


In [ ]:
raw = language.load()
raw['label'] = raw['model'].map(NAME)

# Coverage
short = raw['response_length'] < FLOOR
frame = raw.copy()
frame.loc[short, FORMULAS] = np.nan

# Coverage
frame['aae'] = (frame['fkgl'] - frame['target_grade']).abs()
frame['grade_offset'] = frame['fkgl'] - frame['target_grade']

# Model Names
frame['targeted'] = frame['fkgl'].notna() & frame['target_grade'].notna()

# Withheld Replies
assert raw['fkgl'].notna().all(), 'measuring pass was not run at floor 0'

returned = language.load_texts()
assert len(raw) == len(returned), (
    f'the language layer holds {len(raw):,} rows against {len(returned):,} '
    f'returned replies. Rerun scripts/language.py: the measuring pass has '
    f'included requests a provider withheld.')
assert (raw['response_length'] > 0).all(), (
    'a measured reply has zero words, which only a withheld request does')
print("Minor Replies:", len(raw), int(short.sum()))

# Model Names
def by_scenario(part, measures, keys):
    cell = part.groupby(keys + ['scenario_id', 'condition'],
                        observed=True)[measures].mean()
    scenario = cell.groupby(keys + ['scenario_id'],
                            observed=True)[measures].mean()
    return scenario.groupby(keys, observed=True)[measures].mean()

## Coverage


In [ ]:
# Coverage
coverage = frame.groupby('label').apply(lambda part: pd.Series({
    'Replies': len(part),
    'Median Words': part['response_length'].median(),
    'Below Floor': int((part['response_length'] < FLOOR).sum()),
    'Removed by Floor (%)': round((part['response_length'] < FLOOR).mean() * 100, 1),
    'Median AoA Coverage (%)': round(part['aoa_coverage'].median() * 100, 1),
    'Carrying a Target Grade': int(part['targeted'].sum()),
}), include_groups=False).reindex(ORDER).reset_index().rename(columns={'label': 'Model'})

publish(coverage, 'readability_01_coverage')

print("AoA Coverage:", frame['aoa_coverage'].median(), int(frame['mean_aoa'].isna().sum()))

In [ ]:
# Age Coverage
frame['short'] = (frame['response_length'] < FLOOR) * 100.0
by_age = frame[frame['signal'].eq('stated')].pivot_table(
    index='label', columns='age', values='short').round(1).reindex(ORDER)
by_type = frame.pivot_table(
    index='label', columns='scenario_type', values='short').round(1).reindex(ORDER)
by_age.columns = [f'Age {int(age)}' for age in by_age.columns]
loss = by_age.join(by_type).reset_index().rename(columns={'label': 'Model'})

publish(loss, 'readability_s01_coverage_age')


## Primary Contrast


In [ ]:
# Sampling
def contrast_grades(part, column, left, right):
    """Scenario-level difference between two age blocks.

    Complete case across every age in left + right, not merely across the two
    blocks: a scenario missing a measurable reply at any one of the ages is
    dropped whole. That is stricter than the contrast needs and it is
    deliberate, since a scenario present at seven and absent at nine would
    otherwise weight the minor block towards the ages a model happened to
    answer at length.
    """
    wide = part.pivot_table(index='scenario_id', columns='age',
                            values=column, aggfunc='mean')
    wide = wide.reindex(columns=left + right).dropna()
    return wide[left].mean(axis=1) - wide[right].mean(axis=1)


stated = frame[frame['signal'].eq('stated')]

# Sampling

per_model = {}
rows = []
for label in ORDER:
    diff = contrast_grades(stated[stated['label'] == label], 'fkgl',
                           STATED_MINOR, STATED_ADULT)
    per_model[label] = diff
    point, low, high = bootstrap_paired(diff)
    rows.append({'Model': label, 'Effect (grades)': round(point, 2),
                 'p': permutation_paired(diff), '95% CI Lower': round(low, 2),
                 '95% CI Upper': round(high, 2), 'Complete Scenarios': int(diff.size)})

conditioning = pd.DataFrame(rows)
conditioning['q'] = benjamini_hochberg(conditioning['p'])
conditioning['p'] = conditioning['p'].map(pvalue)
conditioning['q'] = conditioning['q'].map(pvalue)

# Figure
point, low, high = analysis.macro_average(per_model)
conditioning.loc[len(conditioning)] = {
    'Model': MACRO, 'Effect (grades)': round(point, 2), 'p': '', 'q': '',
    '95% CI Lower': round(low, 2), '95% CI Upper': round(high, 2),
    'Complete Scenarios': ''}

conditioning = conditioning[['Model', 'Effect (grades)', 'p', 'q',
                             '95% CI Lower', '95% CI Upper', 'Complete Scenarios']]

publish(conditioning, 'readability_02_conditioning')

print("Model:", point, low)

## Levels


In [ ]:
# Scenario Weighting
levels = by_scenario(stated, ['fkgl'], ['label', 'age'])['fkgl'].unstack()
levels = levels.round(2).reindex(ORDER)
levels.columns = [f'Age {int(age)}' for age in levels.columns]
levels = levels.reset_index().rename(columns={'label': 'Model'})
publish(levels, 'readability_s02_levels')

In [ ]:
# Coverage
targeted = frame[frame['targeted']]
MEASURES = ['fkgl', 'aae', 'grade_offset', 'fre', 'mean_aoa', 'response_length']
calibration = by_scenario(targeted, MEASURES, ['label'])
calibration = calibration.round(2).reindex(ORDER).reset_index()
calibration.columns = ['Model', 'FKGL', 'AAE', 'Grade Offset', 'FRE',
                       'Mean AoA', 'Response Length']
publish(calibration, 'readability_s03_calibration')

## Signal Strength


In [ ]:
# Scenario Weighting
LEVELS = [('Control (No Age)', frame['condition'].eq('neutral')),
          ('Implicit Cue (Adult)', frame['condition'].str.contains('_adult', na=False)),
          ('Explicit Age (Adult)', frame['age'].isin(STATED_ADULT)),
          ('Implicit Cue (Minor)', frame['condition'].str.contains('_minor', na=False)),
          ('Explicit Age (Minor)', frame['age'].isin(STATED_MINOR))]

signals = pd.DataFrame({
    name: by_scenario(frame[mask], ['fkgl'], ['label'])['fkgl'].round(2)
    for name, mask in LEVELS}).reindex(ORDER).reset_index()
signals = signals.rename(columns={'label': 'Model'})
publish(signals, 'readability_s04_signals')

## Robustness


In [ ]:
# Coverage
CUTS = [0, 25, 50, 75, 100]
sensitivity = []
for cut in CUTS:
    part = raw[raw['signal'].eq('stated')].copy()
    part = part[part['response_length'] >= cut]
    effects = [contrast_grades(part[part['label'] == label], 'fkgl',
                               STATED_MINOR, STATED_ADULT).mean()
               for label in ORDER]
    sensitivity.append({'Word Floor': cut,
                        'Replies Retained': int(len(part)),
                        'Macro-average (grades)': round(float(np.mean(effects)), 2),
                        'Models with the Majority Sign':
                            f'{sum(1 for e in effects if e < 0)} of 6'})
publish(pd.DataFrame(sensitivity), 'readability_s05_floor')


### Target Mapping


In [ ]:
# Scenario Weighting
OVERLAP = [7, 9, 11, 13]
coarse = frame[frame['targeted'] & frame['age'].isin(OVERLAP)].copy()
coarse['coarse_target'] = coarse['age'].map(language.coarse_target_grade)
coarse['coarse_aae'] = (coarse['fkgl'] - coarse['coarse_target']).abs()
mapping = by_scenario(coarse, ['aae', 'coarse_aae'], ['label']).round(2)
mapping['Difference'] = (mapping['coarse_aae'] - mapping['aae']).round(2)
mapping = mapping.reindex(ORDER).reset_index()
mapping.columns = ['Model', 'AAE, Continuous', 'AAE, Coarse', 'Difference']
publish(mapping, 'readability_s06_coarse')

### Correlations


In [ ]:
# Correlation
ALL = PRIMARY[:1] + SECONDARY + [
    'gunning_fog', 'ari', 'smog', 'p90_aoa', 'max_aoa', 'difficult_share',
    'aoa_coverage', 'sentence_length', 'word_length', 'ttr', 'mtld']
matrix = frame[ALL].corr().round(2)
matrix.index = [name.replace('_', ' ').title() for name in matrix.index]
matrix.columns = matrix.index
publish(matrix.reset_index().rename(columns={'index': 'Measure'}),
        'readability_s07_correlations')


### Outcome Adjustment


In [ ]:
# Scenario Weighting
verdicts = pd.read_csv(RESULTS_DIR / 'classification.csv', dtype=str)
verdicts = verdicts[(verdicts['answer'] == 'Compliance')
                    & (verdicts['delivery_response'] == 'Yes')]
complied = set(zip(verdicts['prompt_id'], verdicts['model'],
                   verdicts['replicate'].astype(str)))

stated = frame[frame['signal'].eq('stated')].copy()
stated['complied'] = [
    (prompt, model, str(replicate)) in complied
    for prompt, model, replicate in zip(stated['prompt_id'], stated['model'],
                                        stated['replicate'])]

rows = []
for name, subset in [('Benign only',
                      stated[stated['scenario_type'] == 'Benign']),
                     ('Total Compliance only', stated[stated['complied']])]:
    for label in ORDER:
        diff = contrast_grades(subset[subset['label'] == label], 'fkgl',
                               STATED_MINOR, STATED_ADULT)
        if diff.size < 5:
            rows.append({'Restriction': name, 'Model': label,
                         'Effect (grades)': '', '95% CI Lower': '',
                         '95% CI Upper': '',
                         'Complete Scenarios': int(diff.size)})
            continue
        point, low, high = bootstrap_paired(diff)
        rows.append({'Restriction': name, 'Model': label,
                     'Effect (grades)': round(point, 2),
                     '95% CI Lower': round(low, 2),
                     '95% CI Upper': round(high, 2),
                     'Complete Scenarios': int(diff.size)})

outcome = pd.DataFrame(rows)
publish(outcome, 'readability_s08_outcome')

for name in outcome['Restriction'].unique():
    part = outcome[(outcome['Restriction'] == name)
                   & (outcome['Effect (grades)'] != '')]
    if len(part):
        print("Model:", name, part['Effect (grades)'].mean())

### Age Boundary


In [ ]:
# Contrast
rows = []
for label in ORDER:
    diff = contrast_grades(stated[stated['label'] == label], 'fkgl',
                           [17.0], [18.0])
    point, low, high = bootstrap_paired(diff)
    rows.append({'Model': label, 'Effect (grades)': round(point, 2),
                 '95% CI Lower': round(low, 2), '95% CI Upper': round(high, 2),
                 'Complete Scenarios': int(diff.size)})

boundary = pd.DataFrame(rows)
publish(boundary, 'readability_s09_boundary')

# Load Data
for label in ORDER:
    levels = by_scenario(stated[stated['label'] == label], ['fkgl'],
                         ['age'])['fkgl']
    step = levels.get(18.0) - levels.get(17.0)
    span = levels.get(21.0) - levels.get(7.0)
    print("Age Boundary:", label, step)

### Vocabulary Tail


In [ ]:
# Coverage
minor_lexical = frame[frame['signal'].eq('stated')
                      & frame['age'].isin(STATED_MINOR)]
LEXICAL = ['mean_aoa', 'p90_aoa', 'max_aoa', 'difficult_share', 'aoa_coverage']
tail = by_scenario(minor_lexical, LEXICAL, ['label'])
tail = tail.round(3).reindex(ORDER).reset_index()
tail.columns = ['Model', 'Mean AoA', 'P90 AoA', 'Max AoA', 'Difficult Share',
                'AoA Coverage']
publish(tail, 'readability_s10_lexical_tail')

print("Minor Replies:", len(minor_lexical), len(targeted))

In [ ]:
write_captions()
print("Written:", ', '.join((f'{count} {kind}' for kind, count in sorted(analysis.WRITTEN.items()))) + ' written')
